In [181]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.prompts import SystemMessagePromptTemplate,HumanMessagePromptTemplate,ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [182]:
load_dotenv(override=True)

llm = ChatOpenAI(model="gpt-5.4-mini")
llm_groq = ChatGroq(temperature=0,model="openai/gpt-oss-120b",api_key=os.getenv("GROQ_API_KEY"))

parser = StrOutputParser()

In [183]:
technical_system_prompt = SystemMessagePromptTemplate.from_template("""
You are a senior interviewer for the role of {role}.
Write exactly {count} technical interview questions suited to a {level} candidate.
For each question, add a one-line note on what a good answer should include.
Output as a numbered list. No introduction.
""")

In [184]:
hr_behavioural_system_prompt = SystemMessagePromptTemplate.from_template("""
You are an HR interviewer hiring for {role}.
Write exactly {count} behavioural questions suited to a {level} candidate.
Cover teamwork, conflict, deadlines and learning from mistakes.
For each question, add a short hint using the STAR method (Situation, Task, Action, Result).
Output as a numbered list. No introduction.
""")

In [185]:
interviewer_system_prompt = SystemMessagePromptTemplate.from_template("""
You are an HR interviewer hiring for {role}.
Write exactly {count} behavioural questions suited to a {level} candidate.
Cover teamwork, conflict, deadlines and learning from mistakes.
For each question, add a short hint using the STAR method (Situation, Task, Action, Result).
Output as a numbered list. No introduction.
""")

In [186]:
user_propmt = HumanMessagePromptTemplate.from_template("""
Prepare interview material for the role of {role}.
Candidate level: {level}
Number of items: {count}
""")

In [187]:
technical_messages = [technical_system_prompt,user_propmt]
hr_messages = [hr_behavioural_system_prompt,user_propmt]
interviewer_messages = [interviewer_system_prompt,user_propmt]

technical_template = ChatPromptTemplate(technical_messages)
hr_template = ChatPromptTemplate(hr_messages)
interviewer_template = ChatPromptTemplate(interviewer_messages)

technical_chain = technical_template | llm | parser
hr_chain = hr_template | llm | parser
interviewer_chain = interviewer_template | llm | parser

merge_prompt = ChatPromptTemplate.from_messages([
    ("system", "You create a one-page interview prep sheet for a {level} {role} candidate. "
               "Use markdown headings: Technical, Behavioural, Questions to Ask, Final Tips. "
               "Add 3 final tips. Do not add new questions."),
    ("human", "Technical:\n{technical}\n\nBehavioural:\n{hr}\n\nQuestions to ask:\n{interviewer}"),
])

full_chain = (
    RunnablePassthrough.assign(
        technical=technical_chain,
        hr=hr_chain,
        interviewer=interviewer_chain,
    )
    | merge_prompt
    | llm
    | parser
)

# chain = RunnableParallel(
# technical=technical_chain,
# hr=hr_chain,
# interviewer=interviewer_chain | parser
# )


output = full_chain.invoke({"role": "Java backend developer", "level": "senior", "count": 5})
print(output)


# Technical

1. How would you design and optimize a high-throughput Java backend service that handles concurrent requests with low latency?  
   A good answer should include thread management, non-blocking I/O, connection pooling, caching, backpressure, and bottleneck analysis.

2. Explain the Java Memory Model and how it affects thread safety in backend applications.  
   A good answer should include visibility, ordering, atomicity, volatile, synchronization, locks, and examples of common concurrency bugs.

3. How do you approach transaction management and consistency in distributed systems built with Java microservices?  
   A good answer should include ACID vs eventual consistency, saga pattern, idempotency, compensation, distributed transactions, and failure handling.

4. What strategies would you use to diagnose and improve performance issues in a production Java application?  
   A good answer should include profiling, GC analysis, heap/thread dumps, metrics, tracing, log correla

In [188]:
classify_system_prompt = SystemMessagePromptTemplate.from_template("""
Classify the sentiment of this product review.
Reply with exactly one word: positive, negative or neutral.
Review: {review}
""")
classify_human_prompt = HumanMessagePromptTemplate.from_template(
    "Review: {review}"
)
messages = [classify_system_prompt,classify_human_prompt]
classify_template = ChatPromptTemplate.from_messages(messages)

classify_chain = classify_template | llm | parser

review = classify_chain.invoke({
    "review": "I am very disappointed with this product. It stopped working within a week."
})
print(review)

negative


In [189]:
positive_prompt = """
                You are expert in writing reply for positive reviews.
                You need to encourage the user to share their experience on social media.
                Review: {review}
                write email to customer"""
                
positive_template = ChatPromptTemplate.from_template(positive_prompt)
positive_chain = positive_template | llm | parser

negative_prompt = """
You are an expert in writing replies to negative reviews.
You need to apologize sincerely, acknowledge the specific problem the customer mentions,
and invite them to contact customer support so the issue can be solved.
Do not promise a refund or compensation.
Review: {review}
write email to customer"""
negative_template = ChatPromptTemplate.from_template(negative_prompt)
negative_chain = negative_template | llm | parser


neutral_prompt = """
You are an expert in writing replies to neutral reviews.
You need to thank the customer for their feedback, acknowledge what they said,
and ask one short question about what would make the product better for them.
Keep the tone polite and balanced. Do not over-praise or apologize.
Review: {review}
write email to customer"""
neutral_template = ChatPromptTemplate.from_template(neutral_prompt)
neutral_chain = neutral_template | llm | parser

In [190]:
def rout(info):
    if 'positive' in info['sentiment'].lower():
        return positive_chain
    elif 'negative' in info['sentiment'].lower():
        return negative_chain
    else:
        return neutral_chain

In [191]:
def text_lenght(text):
    return len(text)

In [192]:
def text_word_count(text):
    return len(text.split())

In [193]:
full_chain = {"sentiment":classify_chain,'review': lambda x : x['review']} | RunnableLambda(rout) | {'char_counts' : RunnableLambda(text_lenght),'word_count':RunnableLambda(text_word_count),'output':RunnablePassthrough()}

In [194]:
output = full_chain.invoke({"review":"The product is fine. It does the job, but nothing special."})
print(output)

{'char_counts': 322, 'word_count': 58, 'output': 'Subject: Thank you for your feedback\n\nHi,\n\nThank you for sharing your feedback. I’m glad to hear the product does the job for you, and I appreciate your honest thoughts about it being fine but nothing special.\n\nIf you don’t mind me asking, what is one thing that would make the product better for you?\n\nBest,  \n[Your Name]'}
